# Проект спринта 19. Модуль расчёта батч-признаков для платформы интернет-торговли.
## 1. Установка и импорт библиотек

In [ ]:
# Системные библиотеки
import os
from dotenv import load_dotenv
import math
import time

# Библиотека для доступа к СУБД
from sqlalchemy import create_engine, text

# Библиотеки для вычислений и работы с датасетами
import pandas as pd
import numpy as np

# Для форматирования в HTML
from IPython.display import display, HTML, Markdown

### Определение констант

In [ ]:
RUN_DATE = "2025-02-01"

run_date = pd.to_datetime(RUN_DATE)

## 2. Загрузка параметров и соединение с БД

In [ ]:
load_dotenv(".env")

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    os.getenv("DB_USER"),
    os.getenv("DB_PASSWORD"),
    os.getenv("DB_HOST"),
    os.getenv("DB_PORT"),
    os.getenv("DB_NAME")
)

# Создание пула соединений
engine = create_engine(connection_string) 

## 3. Загрузка необработанных данных

In [ ]:
def show_dataset_info(df, title):
    display(HTML(f"<h2>{title}</h2>"))
    df.info()
    display(df.head())
    display(HTML("<h3>Диапазоны дат</h3>"))
    display(pd.DataFrame({"Начальная дата": df.select_dtypes(include='datetime64').min(),
             "Конечная дата": df.select_dtypes(include='datetime64').max()}))
    display(HTML(f"<h3>Количество уникальных идентификаторов</h3>"))
    display(df.filter(like='_id').nunique())

    display(HTML(f"<h3>Количество полных дубликатов: {df.duplicated().sum()}</h3>"))

def load_data():
    with engine.connect() as connection:
        df_events = pd.read_sql_table('events', con=connection)
        df_sessions = pd.read_sql_table('sessions', con=connection)
        df_orders = pd.read_sql_table('orders', con=connection)
        df_customers = pd.read_sql_table('customers', con=connection)

        return df_events, df_sessions, df_orders, df_customers

def load_data_by_run_date(run_date: pd.Timestamp):
    evt_query = text("""
        SELECT customer_id, events.*
        FROM 
            public.events
            INNER JOIN public.sessions USING (session_id)
        WHERE DATE_TRUNC('day', timestamp) BETWEEN :run_date - 30 AND :run_date
    """)
    session_query = text("""
        SELECT *
        FROM public.sessions
        WHERE DATE_TRUNC('day', start_time) BETWEEN :run_date - 30 AND :run_date
    """)
    orders_query = text("""
        SELECT *
        FROM public.orders
        WHERE DATE_TRUNC('day', order_time) BETWEEN :run_date - 30 AND :run_date
    """)

    with engine.connect() as connection:
        df_events = pd.read_sql(evt_query, con=connection, params={"run_date": run_date.date()})
        df_sessions = pd.read_sql(session_query, con=connection, params={"run_date": run_date.date()})
        df_orders = pd.read_sql(orders_query, con=connection, params={"run_date": run_date.date()})

        
        return {
            "events_30": df_events,
            "sessions_30": df_sessions,
            "orders_30": df_orders,
            "events_7": df_events[df_events['timestamp'] >= run_date - pd.Timedelta(days=7)],
            "sessions_7": df_sessions[df_sessions['start_time'] >= run_date - pd.Timedelta(days=7)],
            "orders_7": df_orders[df_orders['order_time'] >= run_date - pd.Timedelta(days=7)]
        }   

df_events, df_sessions, df_orders, df_customers = load_data()

show_dataset_info(df_events, "Таблица Events")
show_dataset_info(df_sessions, "Таблица Sessions")
show_dataset_info(df_orders, "Таблица Orders")
show_dataset_info(df_customers, "Таблица Customers")
data = load_data_by_run_date(run_date)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239089 entries, 0 to 239088
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   event_id      239089 non-null  int64         
 1   session_id    239089 non-null  int64         
 2   timestamp     239089 non-null  datetime64[ns]
 3   event_type    239089 non-null  object        
 4   product_id    214208 non-null  float64       
 5   qty           44908 non-null   float64       
 6   cart_size     14229 non-null   float64       
 7   payment       10652 non-null   object        
 8   discount_pct  10652 non-null   float64       
 9   amount_usd    10652 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(2), object(2)
memory usage: 18.2+ MB


,event_id,session_id,timestamp,event_type,product_id,qty,cart_size,payment,discount_pct,amount_usd
0,11,2,2025-01-31 21:48:42,page_view,929.0,NaN,NaN,None,NaN,NaN
1,12,2,2025-01-31 21:57:42,page_view,226.0,NaN,NaN,None,NaN,NaN
2,13,2,2025-01-31 21:59:07,add_to_cart,226.0,1.0,NaN,None,NaN,NaN
3,14,2,2025-01-31 22:01:42,page_view,864.0,NaN,NaN,None,NaN,NaN
4,15,2,2025-01-31 22:25:42,page_view,1151.0,NaN,NaN,None,NaN,NaN


,Начальная дата,Конечная дата
timestamp,2024-01-01 01:02:40,2025-11-01 01:50:04


event_id      239089
session_id     37699
product_id      1197
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37699 entries, 0 to 37698
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   session_id   37699 non-null  int64         
 1   customer_id  37699 non-null  int64         
 2   start_time   37699 non-null  datetime64[ns]
 3   device       37699 non-null  object        
 4   source       37699 non-null  object        
 5   country      37699 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 1.7+ MB


...

order_id       10652
customer_id     8230
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17846 entries, 0 to 17845
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       17846 non-null  int64         
 1   name              17846 non-null  object        
 2   email             17846 non-null  object        
 3   country           17846 non-null  object        
 4   age               17846 non-null  int64         
 5   signup_date       17846 non-null  datetime64[ns]
 6   marketing_opt_in  17846 non-null  bool          
dtypes: bool(1), datetime64[ns](1), int64(2), object(3)
memory usage: 854.1+ KB


,customer_id,name,email,country,age,signup_date,marketing_opt_in
0,1,Jennifer Salinas,nicholas59@example.org,JP,71,2020-09-04,True
1,2,Phillip Ramos,christinarubio@example.com,IN,26,2020-04-05,False
2,3,Dawn Fowler,jessica03@example.org,BR,21,2023-08-31,True
3,4,Mario Butler,paula27@example.org,FR,63,2022-06-30,True
4,5,Amber Brown,kevin85@example.net,BR,19,2022-07-22,True


,Начальная дата,Конечная дата
signup_date,2020-01-01,2025-10-31


customer_id    17846
dtype: int64

- Записи в таблицах присутствуют для дат 01.01.2024 по 31.10.2025 включительно.
- Даты для формирования 30-дневных срезов - с 31.01.2024.
- Целевая переменная может быть сформирована для данных до 24.10.2025 включительно.
- Пропуски имеются только в таблице `events`

## 4. Создание батч-признаков

In [ ]:
# Предобработка загруженных данных
def preprocess(df_events, df_sessions, df_orders, df_customers):
    
    return df_events, df_sessions, df_orders, df_customers

# Получение 7- и 30-дневных срезов
def get_slices(run_date, df_events, df_sessions, df_orders, df_customers):
    df_dict = {}

    date_from_7 = run_date - pd.Timedelta(days=7)
    date_from_30 = run_date - pd.Timedelta(days=30)

    if date_from_30 < df_events['timestamp'].min():
        raise ValueError("Slice for this run_date is not available")

    # Фильтруем df_customers, чтобы оставить только тех клиентов, которые зарегистрировались 30 и более дней назад
    df_customers_ids = df_customers[df_customers['signup_date'] <= date_from_30][['customer_id']]

    df_dict['events_7'] = (df_events[(df_events['timestamp'] >= date_from_7) &
                                     (df_events['timestamp'] < run_date)]
                           .merge(df_sessions[['session_id', 'customer_id']], on='session_id')
                           .merge(df_customers_ids, on='customer_id', how='inner'))
    
    df_dict['events_30'] = (df_events[(df_events['timestamp'] >= date_from_30) &
                                      (df_events['timestamp'] < run_date)]
                           .merge(df_sessions[['session_id', 'customer_id']], on='session_id')
                           .merge(df_customers_ids, on='customer_id', how='inner'))

    df_dict['sessions_7'] = (df_sessions[(df_sessions['start_time'] >= date_from_7) &
                                        (df_sessions['start_time'] < run_date)]
                            .merge(df_customers_ids, on='customer_id', how='inner'))
    
    df_dict['sessions_30'] = (df_sessions[(df_sessions['start_time'] >= date_from_30) &
                                         (df_sessions['start_time'] < run_date)]
                            .merge(df_customers_ids, on='customer_id', how='inner'))
    
    df_dict['orders_7'] = (df_orders[(df_orders['order_time'] >= date_from_7) &
                                    (df_orders['order_time'] < run_date)]
                            .merge(df_customers_ids, on='customer_id', how='inner'))
    
    df_dict['orders_30'] = (df_orders[(df_orders['order_time'] >= date_from_30) &
                                     (df_orders['order_time'] < run_date)]
                            .merge(df_customers_ids, on='customer_id', how='inner'))

    return df_dict

# Создание батч-признаков
def create_batch_features(df_dict):
    df = {}

    df['page_views_7_count'] = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'page_view']
                                .groupby('customer_id')
                                .size()
                                .rename('page_views_7_count'))

    df['page_views_30_count'] = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'page_view']
                                .groupby('customer_id')
                                .size()
                                .rename('page_views_30_count'))

    df['add_to_cart_7_count'] = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'add_to_cart']
                                .groupby('customer_id')
                                .size()
                                .rename('add_to_cart_7_count'))
    
    df['add_to_cart_30_count'] = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'add_to_cart']
                                .groupby('customer_id')
                                .size()
                                .rename('add_to_cart_30_count'))

    df_purchase_count_7 = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'purchase']
                                .groupby('customer_id')
                                .size()
                                .rename('purchase_count_7'))
        
    df_purchase_count_30 = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'purchase']
                                .groupby('customer_id')
                                .size()
                                .rename('purchase_count_30'))

    df['cart_conv_7'] = (df['add_to_cart_7_count'] / df['page_views_7_count']).rename('cart_conv_7')
    df['cart_conv_30'] = (df['add_to_cart_30_count'] / df['page_views_30_count']).rename('cart_conv_30')
    df['purchase_conv_7'] = (df_purchase_count_7 / df['add_to_cart_7_count']).rename('purchase_conv_7')
    df['purchase_conv_30'] = (df_purchase_count_30 / df['add_to_cart_30_count']).rename('purchase_conv_30')
    df['unique_products_7'] = df_dict['events_7'].groupby('customer_id')['product_id'].nunique().rename('unique_products_7')
    df['unique_products_30'] = df_dict['events_30'].groupby('customer_id')['product_id'].nunique().rename('unique_products_30')

    df_session_durations = (df_dict['sessions_30']
                            .merge(df_dict['events_30']
                                      .groupby('session_id')['timestamp']
                                      .max()
                                      .rename('last_event_time'), 
                                    on='session_id', how='left'))
    df_session_durations['session_duration'] = (df_session_durations['last_event_time'] - 
                                                df_session_durations['start_time']).dt.total_seconds()
    df_session_durations['session_duration'] = df_session_durations['session_duration'].fillna(0)
    
    df['mean_session_duration_30'] = (df_session_durations
                                        .groupby('customer_id')['session_duration']
                                        .mean()
                                        .rename('mean_session_duration_30'))
    df['sessions_count_7'] = df_dict['sessions_7'].groupby('customer_id').size().rename('sessions_count_7')
    df['sessions_count_30'] = df_dict['sessions_30'].groupby('customer_id').size().rename('sessions_count_30')

    df_last_purchase = (
        df_dict['events_7']
        .loc[df_dict['events_7']['event_type'] == 'purchase']
        .groupby('customer_id')['timestamp']
        .max()
    )

    df['days_from_last_purchase'] = (
        (run_date.normalize() - df_last_purchase.dt.normalize()).dt.days + 1
    ).rename('days_from_last_purchase')

    df_dict['orders_30']['total_usd'] = df_dict['orders_30']['total_usd'].fillna(0)
    df['orders_count_30'] = df_dict['orders_30'].groupby('customer_id').size().rename('orders_count_30')
    df['total_usd_sum_30'] = df_dict['orders_30'].groupby('customer_id')['total_usd'].sum().rename('total_usd_sum_30')
    df['total_usd_mean_30'] = df_dict['orders_30'].groupby('customer_id')['total_usd'].mean().rename('total_usd_mean_30')

    # Объединение всех признаков в один DataFrame
    batch_features = pd.concat(df.values(), axis=1)

    # Заполнение пропущенных значений из-за деления на ноль или отсутствия покупок
    batch_features.fillna({'days_from_last_purchase': -1}, inplace=True)
    batch_features.fillna(0, inplace=True)
    
    return batch_features.reset_index()

df_batch = create_batch_features(get_slices(run_date, *preprocess(*load_data())))
display(df_batch.head())

,customer_id,page_views_7_count,page_views_30_count,add_to_cart_7_count,add_to_cart_30_count,cart_conv_7,cart_conv_30,purchase_conv_7,purchase_conv_30,unique_products_7,unique_products_30,mean_session_duration_30,sessions_count_7,sessions_count_30,days_from_last_purchase,orders_count_30,total_usd_sum_30,total_usd_mean_30
0,5,8.0,13,2.0,3.0,0.250000,0.230769,0.5,0.666667,8.0,13,7200.0,1.0,2.0,3.0,2.0,242.49,121.245
1,123,3.0,3,1.0,1.0,0.333333,0.333333,1.0,1.000000,3.0,3,4800.0,1.0,1.0,6.0,1.0,47.41,47.410
2,187,6.0,6,2.0,2.0,0.333333,0.333333,0.5,0.500000,6.0,6,5280.0,1.0,1.0,7.0,1.0,26.89,26.890
3,311,6.0,6,3.0,3.0,0.500000,0.500000,0.0,0.000000,6.0,6,6660.0,1.0,1.0,-1.0,0.0,0.00,0.000
4,317,3.0,3,0.0,0.0,0.000000,0.000000,0.0,0.000000,3.0,3,1740.0,1.0,1.0,-1.0,0.0,0.00,0.000


## 5. Проверка корректности создания признаков

In [ ]:
display(HTML(f"<h3>Количество дубликатов customer_id: {df_batch['customer_id'].duplicated().sum()}</h3>"))
df_batch.info()
df_batch.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1392 entries, 0 to 1391
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               1392 non-null   int64  
 1   page_views_7_count        1392 non-null   float64
 2   page_views_30_count       1392 non-null   int64  
 3   add_to_cart_7_count       1392 non-null   float64
 4   add_to_cart_30_count      1392 non-null   float64
 5   cart_conv_7               1392 non-null   float64
 6   cart_conv_30              1392 non-null   float64
 7   purchase_conv_7           1392 non-null   float64
 8   purchase_conv_30          1392 non-null   float64
 9   unique_products_7         1392 non-null   float64
 10  unique_products_30        1392 non-null   int64  
 11  mean_session_duration_30  1392 non-null   float64
 12  sessions_count_7          1392 non-null   float64
 13  sessions_count_30         1392 non-null   float64
 14  days_fro

,customer_id,page_views_7_count,page_views_30_count,add_to_cart_7_count,add_to_cart_30_count,cart_conv_7,cart_conv_30,purchase_conv_7,purchase_conv_30,unique_products_7,unique_products_30,mean_session_duration_30,sessions_count_7,sessions_count_30,days_from_last_purchase,orders_count_30,total_usd_sum_30,total_usd_mean_30
count,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000,1392.000000
mean,10091.565374,1.049569,4.679598,0.272270,1.254310,0.060447,0.267858,0.047498,0.221374,1.043103,4.652299,4452.939895,0.236351,1.043103,-0.616379,0.313937,43.249684,42.887188
std,5777.935101,2.230367,2.525086,0.741988,1.193071,0.170377,0.258856,0.193148,0.369877,2.218058,2.507460,2469.370453,0.435024,0.216856,1.512266,0.470411,112.706918,112.154399
min,2.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
25%,5157.750000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,2460.000000,0.000000,1.000000,-1.000000,0.000000,0.000000,0.000000
50%,10168.000000,0.000000,5.000000,0.000000,1.000000,0.000000,0.250000,0.000000,0.000000,0.000000,4.000000,4260.000000,0.000000,1.000000,-1.000000,0.000000,0.000000,0.000000
75%,14852.000000,0.000000,7.000000,0.000000,2.000000,0.000000,0.400000,0.000000,0.333333,0.000000,7.000000,6180.000000,0.000000,1.000000,-1.000000,1.000000,33.945000,33.945000
max,19990.000000,11.000000,16.000000,6.000000,7.000000,1.000000,1.000000,1.000000,1.000000,11.000000,16.000000,12180.000000,2.000000,3.000000,8.000000,2.000000,1924.760000,1924.760000
